In [3]:
import pandas as pd

In [4]:
df_sent = pd.read_csv('../Datasets/SI_sent_development.csv', encoding='utf-8')
df_utt = pd.read_csv('../../Utterances/ParlaMint-SI-text.csv', encoding='utf-8', sep='\t')

In [5]:
df_sent.head()

,sent_id,text,annotations,utt_id
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,4.052144,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,3.223034,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",1.778516,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,4.814360,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,3.219104,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1


In [6]:
df_utt.head()

,ID,Text
0,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u1,Spoštovane kolegice poslanke in kolegi poslanc...
1,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u2,Hvala lepa za besedo. Spoštovana predsednica V...
2,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u3,Hvala tudi vam. Dajem besedo predsednici Vlade...
3,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u4,"Spoštovane poslanke in poslanci, lep pozdrav p..."
4,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.u5,Hvala za odgovor. Gospod Prevc bo predstavil z...


## Check for missing utterances in datasets

In [7]:
df1 = set(df_sent['utt_id'])
df2 = set(df_utt['ID'])

count_df1 = len(df1)
count_df2 = len(df2)

print(f"Number of unique utt_id in df1: {count_df1}")
print(f"Number of unique ID in df2: {count_df2}")

# Identify missing or extra IDs
missing_in_df2 = count_df1 - count_df2
extra_in_df2 = count_df2 - count_df1

print(f"IDs in df1 not in df2: {missing_in_df2}")
print(f"IDs in df2 not in df1: {extra_in_df2}")

Number of unique utt_id in df1: 311347
Number of unique ID in df2: 311354
IDs in df1 not in df2: -7
IDs in df2 not in df1: 7


In [8]:
df_utt['ID'] = df_utt['ID'].str.replace(r'(\.[^\.]+)$', r'.ana\1', regex=True)
df_utt.head()

,ID,Text
0,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.ana.u1,Spoštovane kolegice poslanke in kolegi poslanc...
1,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.ana.u2,Hvala lepa za besedo. Spoštovana predsednica V...
2,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.ana.u3,Hvala tudi vam. Dajem besedo predsednici Vlade...
3,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.ana.u4,"Spoštovane poslanke in poslanci, lep pozdrav p..."
4,ParlaMint-SI_2013-03-27-SDZ6-Redna-12.ana.u5,Hvala za odgovor. Gospod Prevc bo predstavil z...


In [9]:
df_sent['exists_in_df2'] = df_sent['utt_id'].isin(df_utt['ID'])
df_utt['exists_in_df1'] = df_utt['ID'].isin(df_sent['utt_id'])

In [10]:
# Ok, so utterances that include only interuptions were removed from the dataset, so the dataset is ok :)
missing_utt = df_utt.loc[df_utt['exists_in_df1'] == False]
missing_utt

,ID,Text,exists_in_df1
6983,ParlaMint-SI_2013-07-12-SDZ6-Redna-16.ana.u12,[[...]] [[tehnične težave]] [[...]],False
12991,ParlaMint-SI_2013-10-21-SDZ6-Redna-18.ana.u104,[[...]] [[oglašanje iz dvorane]],False
74751,ParlaMint-SI_2001-03-20-SDZ3-Redna-04.ana.u93,[[...]] [[Iz klopi - ni vključen.]] [[...]],False
170171,ParlaMint-SI_2010-04-19-SDZ5-Redna-16.ana.u80,[[...]] [[Ni vključen]] [[...]],False
232021,ParlaMint-SI_2011-06-15-SDZ5-Redna-29.ana.u158,[[...]] [[izključen mikrofon]],False
267325,ParlaMint-SI_2005-07-13-SDZ4-Redna-08.ana.u45,[[...]] [[Medsebojno pogovarjanje.]],False
277731,ParlaMint-SI_2002-03-28-SDZ3-Redna-14.ana.u350,[[...]] [[Začel govoriti brez mikrofona.]],False


## Aggregations for the entire ParlaMint-SI corpus


In [11]:
df_sent['length'] = df_sent['text'].apply(lambda x: len(x.split()))

In [12]:
df_sent.head()
aggregated_scores = (
    df_sent.groupby("utt_id")
    .apply(lambda x: (x["annotations"] * x["length"]).sum() / x["length"].sum())
    .reset_index(name="length_avg")
)

df_sent = df_sent.merge(
    aggregated_scores,
    on="utt_id"
)

df_sent.head()

/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_4243/3271136876.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sent.groupby("utt_id")


,sent_id,text,annotations,utt_id,exists_in_df2,length,length_avg
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,4.052144,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,True,9,2.401479
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,3.223034,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,True,8,2.401479
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",1.778516,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,True,87,2.401479
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,4.814360,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,True,4,2.401479
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,3.219104,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,True,3,2.401479


In [13]:
order = ['utt_id', 'length_avg']
df = df_sent[order]


In [14]:
df = df.rename(columns={'utt_id':'ID', 'length_avg':'annotations'})
df.head()

,ID,annotations
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,2.401479
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,2.401479
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,2.401479
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,2.401479
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,2.401479


In [15]:
df = df.drop_duplicates(subset='ID')
df

,ID,annotations
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,2.401479
9,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u2,2.805297
36,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u3,3.365201
39,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u4,3.148236
45,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u5,3.271888
...,...,...
3876151,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u242,3.537891
3876153,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u243,2.805739
3876158,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u244,4.001848
3876160,ParlaMint-SI_2005-01-24-SDZ4-Redna-02.ana.u245,2.460104


In [16]:
df.to_csv('../Datasets/ParlaMint-SI_utt.tsv', sep='\t', encoding='utf-8', index=False)